# 0. Setup & Installation
Run this cell first. Restart runtime if prompted.

In [ ]:
!pip install -U transformers peft bitsandbytes accelerate datasets scikit-learn matplotlib --quiet

# 1. Authentication

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
secret_value_1 = user_secrets.get_secret("HF_TOKEN")

login(token=secret_value_1)

# 2. Configuration

In [ ]:
from dataclasses import dataclass


@dataclass
class InferenceConfig:
    # ---- Model ----
    model_name: str = "Qwen/Qwen3-4B"
    max_seq_length: int = 2048

    # ---- Quantization ----
    load_in_4bit: bool = True
    bnb_4bit_quant_type: str = "nf4"
    use_double_quant: bool = True

    # ---- Prediction marker (plain text, NOT a special token) ----
    prediction_marker: str = "<|prediction|>"

    # ---- Paths ----
    adapter_path: str = "./outputs/final_adapter"


cfg = InferenceConfig()
print(f"Config: {cfg}")

# 3. Load Eval Data

The `text` column already contains the fully formatted prompt with `<|system|>`, `<|conversation|>`, `<|agent|>`, `<|customer|>`, and `<|prediction|>` tokens. We strip the ground-truth label after `<|prediction|>` to build the inference prompt.

In [ ]:
import pandas as pd

eval_data = pd.read_csv("/kaggle/input/notebooks/zygong1994/finetuning-experiment-0326-etl/test.csv")

print(f"Eval: {len(eval_data)} samples")
print(f"Label distribution:\n{eval_data['outcome'].value_counts()}")
print(f"\nSample text (first 500 chars):\n{eval_data['text'].iloc[0][:500]}")

# 4. Build Inference Prompts

Each `text` sample ends with `<|prediction|>0<|/prediction|>` or `<|prediction|>1<|/prediction|>`.
We truncate at (and include) `<|prediction|>` so the model generates the 0/1 token next.

In [ ]:
PREDICTION_MARKER = "<|prediction|>"

def build_inference_prompt(text: str) -> str:
    """Truncate the text at <|prediction|> (inclusive) so the model predicts next token."""
    idx = text.find(PREDICTION_MARKER)
    if idx == -1:
        raise ValueError(f"Prediction marker not found in text: {text[:200]}...")
    return text[:idx + len(PREDICTION_MARKER)]


# Build prompts and extract ground-truth labels
prompts = []
labels = []

for _, row in eval_data.iterrows():
    prompts.append(build_inference_prompt(row["text"]))
    labels.append(int(row["outcome"]))

print(f"Built {len(prompts)} inference prompts")
print(f"\nExample prompt (last 200 chars):\n...{prompts[0][-200:]}")

# 5. Load Model + Tokenizer with LoRA Adapter

Since training used `<|prediction|>` as plain text (not a special token), no embedding resize is needed.
We load the base model in 4-bit, then attach the LoRA adapter.

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# ---- Quantization config ----
bnb_config = BitsAndBytesConfig(
    load_in_4bit=cfg.load_in_4bit,
    bnb_4bit_quant_type=cfg.bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=cfg.use_double_quant,
)

# ---- Load tokenizer (no special tokens were added during training) ----
tokenizer = AutoTokenizer.from_pretrained(cfg.adapter_path)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

# ---- Verify label token IDs ----
label_0_token_id = tokenizer.encode("0", add_special_tokens=False)[0]
label_1_token_id = tokenizer.encode("1", add_special_tokens=False)[0]
print(f"Token IDs -> '0': {label_0_token_id}, '1': {label_1_token_id}")

# ---- Load base model ----
base_model = AutoModelForCausalLM.from_pretrained(
    cfg.model_name,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa",
    torch_dtype=torch.bfloat16,
)

# ---- Load LoRA adapter ----
model = PeftModel.from_pretrained(base_model, cfg.adapter_path)
model.eval()

print(f"Model loaded with LoRA adapter from {cfg.adapter_path}")
print(f"Vocab size: {model.config.vocab_size}")

# 6. Inference: Extract Logit-Based Probabilities

For each sample, we feed the prompt up to `<|prediction|>` and read the logits at the last position.
We extract logits for tokens "0" and "1", apply softmax to get `P(convert)`, and also do greedy generation for a few tokens to show the raw model output.

In [ ]:
@torch.no_grad()
def get_conversion_probability(model, tokenizer, prompt: str):
    """
    Feed prompt through the model, extract logits at the last position,
    and compute P(1) vs P(0) via softmax over the two label tokens.
    """
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                       max_length=cfg.max_seq_length).to(model.device)
    outputs = model(**inputs)
    logits = outputs.logits[:, -1, :]  # (1, vocab_size)

    binary_logits = logits[:, [label_0_token_id, label_1_token_id]]  # (1, 2)
    probs = F.softmax(binary_logits, dim=-1)

    return {
        "prob_0": probs[0, 0].item(),
        "prob_1": probs[0, 1].item(),
        "logit_0": logits[0, label_0_token_id].item(),
        "logit_1": logits[0, label_1_token_id].item(),
    }


# ---- Quick test on first sample ----
test_result = get_conversion_probability(model, tokenizer, prompts[0])
print(f"Sample 0 | Label: {labels[0]}")
print(f"  P(0)={test_result['prob_0']:.4f}  P(1)={test_result['prob_1']:.4f}")
print(f"  Logit(0)={test_result['logit_0']:.3f}  Logit(1)={test_result['logit_1']:.3f}")

# 7. Run Inference on Full Eval Set

In [ ]:
from tqdm import tqdm

results = []

print("Running inference on eval set...")
for i in tqdm(range(len(prompts)), desc="Inference"):
    res = get_conversion_probability(model, tokenizer, prompts[i])
    res["ground_truth"] = labels[i]
    res["predicted_class"] = 1 if res["prob_1"] > 0.5 else 0
    results.append(res)

results_df = pd.DataFrame(results)
print(f"\nDone. {len(results_df)} samples processed.")
results_df.head(10)

# 8. Classification Metrics

In [ ]:
from sklearn.metrics import (
    accuracy_score, roc_auc_score, classification_report,
    confusion_matrix, log_loss
)
import numpy as np

y_true = results_df["ground_truth"].values
y_pred = results_df["predicted_class"].values
y_prob = results_df["prob_1"].values

accuracy = accuracy_score(y_true, y_pred)
auc_roc = roc_auc_score(y_true, y_prob)
sklearn_ce = log_loss(y_true, y_prob)

print("=" * 60)
print("CLASSIFICATION METRICS")
print("=" * 60)
print(f"Accuracy:       {accuracy:.4f}")
print(f"AUC-ROC:        {auc_roc:.4f}")
print(f"Cross-Entropy:  {sklearn_ce:.4f}")
print()
print(classification_report(y_true, y_pred, target_names=["No Convert (0)", "Convert (1)"]))
print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

# 9. Cross-Entropy Analysis (Per-Sample)

Binary cross-entropy per sample: $H_i = -[y_i \log p_i + (1 - y_i) \log(1 - p_i)]$

In [ ]:
eps = 1e-7
y_prob_clipped = np.clip(y_prob, eps, 1.0 - eps)

per_sample_ce = -(y_true * np.log(y_prob_clipped) + (1 - y_true) * np.log(1 - y_prob_clipped))

results_df["cross_entropy"] = per_sample_ce

print("=" * 60)
print("PER-SAMPLE CROSS-ENTROPY STATISTICS")
print("=" * 60)
print(f"Mean:  {per_sample_ce.mean():.4f}")
print(f"Std:   {per_sample_ce.std():.4f}")
print(f"Min:   {per_sample_ce.min():.4f}")
print(f"Max:   {per_sample_ce.max():.4f}")
print(f"Median:{np.median(per_sample_ce):.4f}")

ce_class_0 = per_sample_ce[y_true == 0]
ce_class_1 = per_sample_ce[y_true == 1]
print(f"\nPer-class mean cross-entropy:")
print(f"  No Convert (0): {ce_class_0.mean():.4f} (n={len(ce_class_0)})")
print(f"  Convert    (1): {ce_class_1.mean():.4f} (n={len(ce_class_1)})")

# 10. Plots & Diagrams

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, precision_recall_curve, auc
from sklearn.calibration import calibration_curve

fig, axes = plt.subplots(2, 3, figsize=(18, 11))

# ---- 1. Histogram of predicted probabilities by class ----
ax = axes[0, 0]
ax.hist(y_prob[y_true == 0], bins=20, alpha=0.6, label="No Convert (0)", color="tab:blue", edgecolor="black")
ax.hist(y_prob[y_true == 1], bins=20, alpha=0.6, label="Convert (1)", color="tab:orange", edgecolor="black")
ax.axvline(x=0.5, color="red", linestyle="--", label="Threshold=0.5")
ax.set_xlabel("P(Convert)")
ax.set_ylabel("Count")
ax.set_title("Predicted Probability Distribution")
ax.legend()

# ---- 2. ROC Curve ----
ax = axes[0, 1]
fpr, tpr, _ = roc_curve(y_true, y_prob)
roc_auc_val = auc(fpr, tpr)
ax.plot(fpr, tpr, color="tab:blue", lw=2, label=f"AUC = {roc_auc_val:.3f}")
ax.plot([0, 1], [0, 1], color="gray", linestyle="--", lw=1)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve")
ax.legend(loc="lower right")

# ---- 3. Precision-Recall Curve ----
ax = axes[0, 2]
precision, recall, _ = precision_recall_curve(y_true, y_prob)
pr_auc_val = auc(recall, precision)
ax.plot(recall, precision, color="tab:green", lw=2, label=f"PR AUC = {pr_auc_val:.3f}")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curve")
ax.legend(loc="lower left")

# ---- 4. Per-sample cross-entropy histogram by class ----
ax = axes[1, 0]
ax.hist(ce_class_0, bins=20, alpha=0.6, label="No Convert (0)", color="tab:blue", edgecolor="black")
ax.hist(ce_class_1, bins=20, alpha=0.6, label="Convert (1)", color="tab:orange", edgecolor="black")
ax.set_xlabel("Cross-Entropy")
ax.set_ylabel("Count")
ax.set_title("Per-Sample Cross-Entropy Distribution")
ax.legend()

# ---- 5. Calibration Curve ----
ax = axes[1, 1]
prob_true, prob_pred = calibration_curve(y_true, y_prob, n_bins=10, strategy="uniform")
ax.plot(prob_pred, prob_true, marker="o", color="tab:purple", lw=2, label="Model")
ax.plot([0, 1], [0, 1], color="gray", linestyle="--", lw=1, label="Perfectly Calibrated")
ax.set_xlabel("Mean Predicted Probability")
ax.set_ylabel("Fraction of Positives")
ax.set_title("Calibration Curve")
ax.legend(loc="lower right")

# ---- 6. Confusion Matrix Heatmap ----
ax = axes[1, 2]
cm = confusion_matrix(y_true, y_pred)
im = ax.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["Pred 0", "Pred 1"])
ax.set_yticklabels(["True 0", "True 1"])
ax.set_title("Confusion Matrix")
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=16)
fig.colorbar(im, ax=ax)

plt.tight_layout()
plt.savefig("inference_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved inference_results.png")

# 11. Sample Generations (Greedy Decode)

Generate a few tokens after `<|prediction|>` to see the raw model output.
Expected: `0` or `1` followed by `<|/prediction|>` (or reasoning text once that is trained).

In [ ]:
@torch.no_grad()
def generate_prediction(model, tokenizer, prompt: str, max_new_tokens=50):
    """Greedy-decode a few tokens after the prompt to inspect raw output."""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                       max_length=cfg.max_seq_length).to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=1.0,
        pad_token_id=tokenizer.pad_token_id,
    )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=False)


# Show first 5 samples
n_show = 5
print("=" * 60)
print("SAMPLE GENERATIONS")
print("=" * 60)
for i in range(n_show):
    generated = generate_prediction(model, tokenizer, prompts[i])
    print(f"\nSample {i} | Ground truth: {labels[i]} | P(1): {results[i]['prob_1']:.4f}")
    print(f"  Generated: {generated.strip()}")

# 12. Save Results

In [ ]:
results_df.to_csv("inference_results.csv", index=False)
print(f"Results saved to inference_results.csv")
print(f"\nSummary:")
print(f"  Accuracy:      {accuracy:.4f}")
print(f"  AUC-ROC:       {auc_roc:.4f}")
print(f"  Cross-Entropy: {sklearn_ce:.4f}")
print(f"  Samples:       {len(results_df)}")